<a href="https://colab.research.google.com/github/AdarshSRM/FirstSite/blob/main/Ytdlp_bulk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:

#@title 🛠️ STEP 1: Setup & Submit Data { display-mode: "form" }
COOKIES_DATA = "" #@param {type:"string"}
#@markdown Paste the entire content of cookies.txt above if required by the site.

import os
import re

print("Installing yt-dlp[default] and dependencies...")
!pip install -U "yt-dlp[default]" > /dev/null

# RECONSTRUCT COOKIES
cleaned_data = COOKIES_DATA.replace('\r', '').replace('\n', ' ')
# Updated regex to handle standard cookie domains including motherless
fixed_content = re.sub(r'(\.(youtube|motherless|[\w-]+)\.com)', r'\n\1', cleaned_data)

if not fixed_content.startswith('# Netscape'):
    fixed_content = "# Netscape HTTP Cookie File\n" + fixed_content

final_cookies = []
for line in fixed_content.split('\n'):
    if line.strip():
        tabbed_line = re.sub(r'\s{2,}', '\t', line.strip())
        final_cookies.append(tabbed_line)

with open('cookies.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(final_cookies))

# CONFIGURE RUNTIME
os.makedirs(os.path.expanduser('~/.config/yt-dlp'), exist_ok=True)
with open(os.path.expanduser('~/.config/yt-dlp/config'), 'w') as f:
    f.write("--js-runtimes node\n")
    f.write("--remote-components ejs:github\n")
    f.write('--extractor-args "youtube:player_client=default,-android_sdkless"\n')

print("\n✅ SYSTEM READY")
print(f"Cookies: {len(final_cookies)} lines reconstructed.")

Installing yt-dlp[default] and dependencies...

✅ SYSTEM READY
Cookies: 1 lines reconstructed.


In [22]:

#@title 🚀 STEP 2: Download Playlist (Fixed for Duplicate Titles)
URL = "https://motherless.com/G863CA08" #@param {type:"string"}

import yt_dlp
import os
import shutil
from google.colab import files

# Prepare the environment
output_dir = "images"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# UPDATED: outtmpl now includes %(id)s to prevent overwriting files with the same title
ydl_opts = {
    'cookiefile': 'cookies.txt',
    'outtmpl': f'{output_dir}/%(title)s_%(id)s.%(ext)s',
    'merge_output_format': 'mp4',
    'noplaylist': True,
    'quiet': True,
    'no_warnings': True
}

print(f"Fetching playlist info for: {URL}")

try:
    # First extraction to get playlist entries
    with yt_dlp.YoutubeDL({'extract_flat': 'in_playlist', 'cookiefile': 'cookies.txt', 'quiet': True}) as ydl_extract:
        playlist_info = ydl_extract.extract_info(URL, download=False)

        # Handle cases where URL is a single video or a playlist
        if 'entries' in playlist_info:
            entries_list = list(playlist_info['entries'])
        else:
            entries_list = [playlist_info]

        print(f"Found {len(entries_list)} items. Starting checks...\n")

        for entry in entries_list:
            if not entry:
                continue

            video_url = entry.get('url') or entry.get('webpage_url')
            # Extract full info for each entry to verify formats
            with yt_dlp.YoutubeDL({'cookiefile': 'cookies.txt', 'quiet': True}) as ydl_check:
                video_info = ydl_check.extract_info(video_url, download=False)
                title = video_info.get('title', 'Unknown')
                formats = video_info.get('formats', [])

                # Format check logic
                if len(formats) > 1:
                    raise Exception(f"ERROR: {len(formats)} formats found for '{title}'. Aborting.")

                fmt_id = formats[0].get('format_id', '0') if formats else '0'
                print(f"✅ 1 format confirmed (ID: {fmt_id}). Downloading: {title}")

                # Download with the specific format and the ID-inclusive template
                current_opts = ydl_opts.copy()
                current_opts['format'] = fmt_id
                with yt_dlp.YoutubeDL(current_opts) as ydl_download:
                    ydl_download.download([video_url])

    # Finalizing
    if os.listdir(output_dir):
        print("\n📦 Zipping files...")
        shutil.make_archive(output_dir, 'zip', output_dir)

        print("⬇️ Triggering download...")
        files.download(f"{output_dir}.zip")
    else:
        print("\n⚠️ No files were downloaded.")

except Exception as e:
    print(f"\n❌ {e}")

Fetching playlist info for: https://motherless.com/G863CA08
Found 7 items. Starting checks...

✅ 1 format confirmed (ID: 0). Downloading: Mariana Diarco - Jugala.avi
✅ 1 format confirmed (ID: 0). Downloading: Solange Gomez - Jugala.avi
✅ 1 format confirmed (ID: 0). Downloading: Floppy Tesouro - Jugala.avi
✅ 1 format confirmed (ID: 0). Downloading: Erika Midtank - Jugala.avi
✅ 1 format confirmed (ID: 0). Downloading: Andrea Estevez - Jugala.avi
✅ 1 format confirmed (ID: 0). Downloading: Debora Natalin Pistarchi - Jugala.avi
✅ 1 format confirmed (ID: 0). Downloading: Andrea Rincon - Jugala.avi

📦 Zipping files...
⬇️ Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>